# One Euro / Exponential Filter Equivalence Check
Upload your two CSVs and run all cells.

In [ ]:
import pandas as pd
import numpy as np

# Upload files (works in Colab)
try:
    from google.colab import files
    print("Upload One Euro CSV:")
    up = files.upload()
    oneeuro_file = list(up.keys())[0]
    print("\nUpload Exponential CSV:")
    up2 = files.upload()
    exp_file = list(up2.keys())[0]
except:
    # Local: put your file paths here
    oneeuro_file = "oneeuro-2026-02-09T18-59-52-485Z.csv"
    exp_file = "exponential-2026-02-09T18-59-53-448Z.csv"

oe = pd.read_csv(oneeuro_file)
ex = pd.read_csv(exp_file)
oe.columns = oe.columns.str.strip()
ex.columns = ex.columns.str.strip()
print(f"One Euro: {len(oe)} rows, Exponential: {len(ex)} rows")

In [ ]:
# === THE MATH ===
fs = 60  # sampling rate (Hz)
Te = 1 / fs  # sampling period (seconds)

def alpha_from_fc(fc):
    """One Euro minCutoff -> equivalent exponential alpha (when velocity=0)"""
    tau = 1 / (2 * np.pi * fc)
    return 1 / (1 + tau / Te)

def fc_from_alpha(a):
    """Exponential alpha -> equivalent One Euro minCutoff"""
    return a / (2 * np.pi * Te * (1 - a))

print(f"Sampling rate = {fs} Hz, Te = {Te:.6f} s ({Te*1000:.2f} ms)")
print(f"\nFormula: alpha = 1 / (1 + 1/(2*pi*fc_min*Te))")

In [ ]:
# === EQUIVALENCE CHECK ===
# For each minCutoff (with smallest beta), compare to exponential at equivalent alpha

min_beta = oe['beta'].min()
min_dcutoff = oe[oe['beta'] == min_beta]['dCutoff'].min()
oe_base = oe[(oe['beta'] == min_beta) & (oe['dCutoff'] == min_dcutoff)].copy()
oe_base['equiv_alpha'] = oe_base['minCutoff'].apply(alpha_from_fc)

# Interpolate exponential variance at each equivalent alpha
exp_interp = np.interp(oe_base['equiv_alpha'].values,
                        ex['alpha'].values,
                        ex['meanVariance'].values)
exp_lat_interp = np.interp(oe_base['equiv_alpha'].values,
                            ex['alpha'].values,
                            ex['meanLatency'].values)

oe_base = oe_base.copy()
oe_base['exp_var_interp'] = exp_interp
oe_base['exp_lat_interp'] = exp_lat_interp
oe_base['var_diff_pct'] = abs(oe_base['meanVariance'] - oe_base['exp_var_interp']) / oe_base['meanVariance'] * 100

result = oe_base[['minCutoff','equiv_alpha','meanVariance','exp_var_interp','var_diff_pct','meanLatency','exp_lat_interp']].copy()
result.columns = ['fc_min','equiv_alpha','1E_var','exp_var','diff_%','1E_lat','exp_lat']
result = result.round({'equiv_alpha':6,'1E_var':2,'exp_var':2,'diff_%':1,'1E_lat':0,'exp_lat':0})

print(f"Comparing One Euro (beta={min_beta}, dCutoff={min_dcutoff}) vs Exponential at equivalent alpha:\n")
print(result.to_string(index=False))
print(f"\nAverage variance difference: {result['diff_%'].mean():.1f}%")
print(f"Matches within 5%: {(result['diff_%'] < 5).sum()} / {len(result)}")

In [ ]:
# === PLOT: Variance equivalence ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter - should be on diagonal
ax = axes[0]
colors = ['green' if d < 5 else 'orange' if d < 15 else 'red' for d in result['diff_%']]
ax.scatter(result['exp_var'], result['1E_var'], c=colors, s=80, zorder=3, edgecolors='black', linewidth=0.5)
lim = max(result['exp_var'].max(), result['1E_var'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Perfect match (y=x)')
ax.set_xlabel('Exponential Variance (px)')
ax.set_ylabel('One Euro Variance (px)')
ax.set_title('Variance Equivalence Check')
ax.legend()
for _, r in result.iterrows():
    ax.annotate(f'fc={r["fc_min"]}', (r['exp_var'], r['1E_var']), fontsize=7, alpha=0.7)

# Plot 2: % difference by fc_min
ax = axes[1]
ax.bar(range(len(result)), result['diff_%'], color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='5% threshold')
ax.set_xticks(range(len(result)))
ax.set_xticklabels([f'{fc:.3f}' for fc in result['fc_min']], rotation=45, fontsize=8)
ax.set_xlabel('minCutoff (Hz)')
ax.set_ylabel('Variance Difference (%)')
ax.set_title('How close is the match?')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# === RED FLAG TEST ===
# Does variance change when beta/dCutoff change but minCutoff stays the same?

red_flag = oe.groupby('minCutoff')['meanVariance'].agg(['mean','std','min','max','count']).reset_index()
red_flag['range'] = red_flag['max'] - red_flag['min']
red_flag['cv_%'] = (red_flag['std'] / red_flag['mean'] * 100).round(1)
red_flag['flag'] = red_flag['cv_%'].apply(lambda x: 'RED FLAG' if x > 5 else 'OK')

print("Red Flag Test: Does variance change when only beta/dCutoff change?")
print("(It shouldn't if data is truly stationary)\n")
print(red_flag.round(2).to_string(index=False))
print(f"\nRed flags: {(red_flag['cv_%'] > 5).sum()} / {len(red_flag)}")
print("\nExplanation: MediaPipe noise creates non-zero derivatives even when")
print("'stationary'. Higher beta amplifies this noise -> higher effective cutoff -> higher variance.")

In [ ]:
# === PARETO FRONTS OVERLAY ===

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(oe['meanVariance'], oe['meanLatency'], c='red', alpha=0.15, s=10, label=f'One Euro (all {len(oe)} combos)')
ax.scatter(ex['meanVariance'], ex['meanLatency'], c='blue', alpha=0.15, s=10, label=f'Exponential (all {len(ex)} alphas)')

# Highlight equivalent pairs
for _, r in oe_base.iterrows():
    ax.plot([r['meanVariance'], r['exp_var_interp']],
            [r['meanLatency'], r['exp_lat_interp']],
            'k-', alpha=0.3, linewidth=0.8)

ax.scatter(oe_base['meanVariance'], oe_base['meanLatency'], c='red', s=60, zorder=5, edgecolors='black', label='1E at equiv alpha')
ax.scatter(oe_base['exp_var_interp'], oe_base['exp_lat_interp'], c='blue', s=60, zorder=5, edgecolors='black', marker='D', label='Exp at equiv alpha')

ax.set_xlabel('Variance (px) - lower is better')
ax.set_ylabel('Latency (ms) - lower is better')
ax.set_title('Pareto Fronts with Equivalent Parameter Pairs')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === QUICK REFERENCE TABLE ===
ref_fc = [0.001, 0.01, 0.021, 0.041, 0.061, 0.081, 0.101, 0.201, 0.5, 1.0]
ref = pd.DataFrame({
    'fc_min (Hz)': ref_fc,
    'equiv_alpha': [alpha_from_fc(f) for f in ref_fc],
    'tau (s)': [1/(2*np.pi*f) for f in ref_fc]
}).round(6)
print(f"Reference: alpha <-> minCutoff at {fs} Hz\n")
print(ref.to_string(index=False))